# Rodin-1B Base → 1B puissant parfaitement français — Fine-tune QLoRA ULTRA MAX v3

Base : `rodin-llm/rodin-1b` 1.238B, LLaMA-style 22 layers 2048 hidden 64k vocab FR, 32B tokens déjà vus, Apache 2.0, FrenchBench vocab 0.773 grammaire 0.765

Objectif : continue pretraining 2B tokens FR haute qualité + SFT FR → 1B puissant parfaitement français

Optimisations v3 MAX : QLoRA 4-bit + LoRA r=64 (50M trainables), FlashAttention SDPA, gradient checkpointing, compile max-autotune fullgraph, cudnn benchmark, TF32, prefetch 2, DDP dual-T4 no_sync +15-30%, bucket 10 MB +5%, FP16 compress +10-20% PCIe, static_graph +2-5%, fused AdamW +5-10%, foreach +2%, SwiGLU fused +5-10%

Temps : 2B tokens LoRA = 50-60h dual-T4 = 5 sessions Kaggle 12h avec resume. Full 1B 2B tokens = 100h. 10B tokens = 250-300h impossible T4, besoin B200.


In [ ]:
!pip install transformers accelerate peft bitsandbytes trl datasets tokenizers -q
!git clone https://github.com/Snowoo-2z/Llm-search.git /tmp/Llm-search || true
%cd /tmp/Llm-search
!ls scripts/finetune_rodin1b.py


In [ ]:
# Test chargement Rodin-1B base
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
tok = AutoTokenizer.from_pretrained('rodin-llm/rodin-1b')
print(tok.vocab_size, tok.eos_token)
model = AutoModelForCausalLM.from_pretrained('rodin-llm/rodin-1b', torch_dtype=torch.float16, device_map='auto', load_in_4bit=True)
print(model)
prompt = 'La photosynthèse est'
ids = tok(prompt, return_tensors='pt').to(model.device)
out = model.generate(**ids, max_new_tokens=50, temperature=0.7)
print(tok.decode(out[0], skip_special_tokens=True))


In [ ]:
# Build dataset FR 2B tokens pour continue pretraining
# FineWeb2-HQ fra_Latn filtré qualité 0.6 + Pleias
!MAX_MB=2000 MIN_QUALITY_SCORE=0.6 OUT=data/fr_2b VOCAB=64000 SEQ_LEN=2048 TOKENIZER_CHARS=20000000 BLOOM=1 bash scripts/build_fr_optimized_v3.sh


In [ ]:
# Continue pretraining LoRA 2B tokens — mono-T4 (50-60h dual-T4 = 100h mono)
# 2B / (2*8*2048) = 61035 steps dual-T4, 122070 steps mono-T4
!python scripts/finetune_rodin1b.py --model rodin-llm/rodin-1b --data data/fr_2b --out runs/rodin1b_cont_2b --batch-size 2 --grad-accum 8 --seq-len 2048 --max-steps 122070 --lora-r 64 --lora-alpha 128 --lr 2e-4 --fp16 --gradient-checkpointing --compile --compile-mode max-autotune --compile-fullgraph --cudnn-benchmark --tf32 --prefetch 2 --eval-interval 1000 --save-interval 1000 --resume


In [ ]:
# Continue pretraining LoRA 2B tokens — dual-T4 Kaggle ULTRA MAX v3 — 50-60h = 5 sessions 12h
!torchrun --nproc_per_node=2 scripts/finetune_rodin1b.py --model rodin-llm/rodin-1b --data data/fr_2b --out runs/rodin1b_cont_2b --batch-size 2 --grad-accum 8 --seq-len 2048 --max-steps 61035 --lora-r 64 --lora-alpha 128 --lr 2e-4 --fp16 --gradient-checkpointing --compile --compile-mode max-autotune --compile-fullgraph --cudnn-benchmark --tf32 --prefetch 2 --bucket-cap-mb 10 --ddp --ddp-fp16-compress --ddp-static-graph --eval-interval 1000 --save-interval 1000 --resume


In [ ]:
# SFT français après continue pretraining
# Données : Aya FR + French Instruct + Claire + 88milSMS transcodé
!python scripts/sft_rodin1b.py --model runs/rodin1b_cont_2b/final --data data/sft_fr --out runs/rodin1b_instruct_fr --lora-r 64 --lr 1e-4 --epochs 3


In [ ]:
# Test final
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch
base = AutoModelForCausalLM.from_pretrained('rodin-llm/rodin-1b', torch_dtype=torch.float16, device_map='auto', load_in_4bit=True)
model = PeftModel.from_pretrained(base, 'runs/rodin1b_instruct_fr')
tok = AutoTokenizer.from_pretrained('rodin-llm/rodin-1b')
prompt = 'Explique la photosynthèse en français :'
ids = tok(prompt, return_tensors='pt').to(model.device)
out = model.generate(**ids, max_new_tokens=100, temperature=0.7)
print(tok.decode(out[0], skip_special_tokens=True))
# Test abréviations
prompt2 = 'slt cv ? mdr'
ids2 = tok(prompt2, return_tensors='pt').to(model.device)
out2 = model.generate(**ids2, max_new_tokens=50)
print(tok.decode(out2[0]))
